# H₂O VQE: asymmetric O–H bond extension

This notebook reproduces the small-system chemistry reference and an ideal-statevector PUCCSD-VQE run from the Quanta H₂O VQE repository.

**Model:** H₂O/STO-3G; H–O–H = 104.5°; one O–H bond fixed at 0.96 Å and the second at 2.20 Å; frozen-core CAS(8e,6o).

The VQE is compared with active-space CASCI, not full-basis FCI. The notebook uses a classical statevector simulator; it does not submit jobs to IBM hardware or require an API token. The VQE cell can take several minutes.


In [ ]:
%pip install --prefer-binary "git+https://github.com/Athleity/Quanta-H2O_VQE-.git"


## 1. Compute and validate the chemistry references

This runs PySCF RHF, all-electron STO-3G FCI, frozen-core CASCI, and an independent mapped-Hamiltonian exact check.


In [ ]:
!h2o-vqe --reference-only --stretched-oh 2.20 --output results/notebook/reference-r2.20


## 2. Run PUCCSD(3)-VQE

The circuit starts from the active-space Hartree–Fock state. SLSQP uses the exact statevector objective and the run is capped at 100 optimizer iterations. The stored parameter ordering follows the ansatz parameter order.


In [ ]:
!h2o-vqe --methods puccsd --ansatz-reps 3 --optimizer slsqp --restarts 1 --seed 20261023 --maxiter 100 --output results/notebook/puccsd3-r2.20


## 3. Read the saved run and report its error

The output is read from the run JSON instead of hard-coding an expected energy. This also makes it clear if a rerun finishes at a different optimizer point.


In [ ]:
from pathlib import Path
import json

run_dir = Path("results/notebook/puccsd3-r2.20")
reference = json.loads((run_dir / "reference.json").read_text())
summary = json.loads((run_dir / "summary.json").read_text())
run = next(row for row in summary["vqe_runs"] if row["method"] == "puccsd")
casci = reference["references"]["frozen_core_casci_hartree"]
energy = run["total_energy_hartree"]
error_mha = abs(energy - casci) * 1000
assert run["ansatz_reps"] == 3
assert len(run["optimal_point"]) == run["num_parameters"]
mapping = reference["mapping"]
print(f"Mapped qubits: {mapping['mapped_qubits']}")
print(f"Parameters: {run['num_parameters']}")
print(f"Energy: {energy:.12f} Ha")
print(f"Active-space CASCI: {casci:.12f} Ha")
print(f"Absolute error: {error_mha:.4f} mHa")
print(f"Within 1.6 mHa: {error_mha < 1.6}")

## 4. Plot the reference error

The 1.6 mHa line is the project’s chemical-accuracy threshold relative to active-space CASCI. It is not an error bar.


In [ ]:
import matplotlib.pyplot as plt

rhf = reference["references"]["rhf_hartree"]
rhf_error_mha = abs(rhf - casci) * 1000
fig, ax = plt.subplots(figsize=(7.2, 4.6), constrained_layout=True)
labels = ["RHF", "PUCCSD ×3"]
values = [rhf_error_mha, error_mha]
bars = ax.bar(labels, values, color=["#0072B2", "#E69F00"], width=0.58)
ax.axhline(1.6, color="#009E73", linestyle="--", linewidth=1.8, label="1.6 mHa target")
ax.set_yscale("symlog", linthresh=1.0)
ax.set_ylabel("Absolute energy error vs active-space CASCI (mHa)")
ax.set_title("H₂O asymmetric stretch · STO-3G · R₂ = 2.20 Å")
ax.grid(axis="y", alpha=0.22)
ax.legend(frameon=False)
for bar, value in zip(bars, values):
    ax.annotate(f"{value:.3f} mHa", (bar.get_x() + bar.get_width()/2, value),
                xytext=(0, 6), textcoords="offset points", ha="center", va="bottom", fontsize=9)
plt.show()


## Interpretation

The active-space calculation is exact only for the selected CAS(8e,6o) Hamiltonian and chosen orbitals. The full STO-3G molecule has a different FCI reference. Results in this notebook are noiseless classical simulation results; finite-shot and IBM QPU measurements are separate experiments and should be reported with their own uncertainty, backend, and circuit metadata.
